# LA Studio forced-alignment — Wav2Vec2 Chinese Aligner

This notebook loads exactly `wav2vec2-aligner-zh` (`cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF`) on CUDA.
It does not use API Gateway and refuses every other model ID.

1. Choose **Runtime → Change runtime type → GPU**.
2. Run all cells.
3. Copy the printed URL and token into LA Studio.


In [ ]:
!nvidia-smi
!git clone --quiet --recursive https://github.com/CrispStrobe/CrispASR.git /content/CrispASR
!git -C /content/CrispASR checkout --quiet 754b67289cf1137e3ed722885705f94132fc614f
!git -C /content/CrispASR submodule update --init --recursive
!cmake -S /content/CrispASR -B /content/CrispASR/build -G Ninja -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON
!cmake --build /content/CrispASR/build --target crispasr -j2
%pip install -q "fastapi==0.115.12" "uvicorn==0.34.3" "python-multipart==0.0.20"

!wget -q --show-progress -O /content/wav2vec2-aligner-zh-q4_k.gguf https://huggingface.co/cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF/resolve/main/wav2vec2-large-xlsr-53-chinese-zh-cn-q4_k.gguf


In [ ]:
from pathlib import Path

WORKER = Path('/content/la_studio_alignment_worker.py')
WORKER.write_text('import os\nimport re\nimport subprocess\nimport tempfile\nimport threading\nfrom pathlib import Path\n\nimport torch\nfrom fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile\n\nif not torch.cuda.is_available():\n    raise RuntimeError("CUDA is unavailable. In Colab choose Runtime > Change runtime type > GPU, then Run all.")\n\nimport json\n\nMODEL_ID = "wav2vec2-aligner-zh"\nMODEL_NAME = "Wav2Vec2 Chinese Aligner"\nUPSTREAM_MODEL = "cstr/wav2vec2-large-xlsr-53-chinese-zh-cn-GGUF"\nSUPPORTED_LANGUAGES = ["zh", "zho", "chi", "cmn", "zh-cn"]\nCRISPASR = "/content/CrispASR/build/bin/crispasr"\nALIGNER_MODEL = "/content/wav2vec2-aligner-zh-q4_k.gguf"\n\nif not Path(CRISPASR).is_file() or not Path(ALIGNER_MODEL).is_file():\n    raise RuntimeError("CrispASR CUDA runtime or the exact aligner model is missing")\n\ndef _collect_crisp_segments(payload):\n    if isinstance(payload, list):\n        entries = payload\n    elif isinstance(payload, dict):\n        entries = payload.get("segments", payload.get("alignment", payload.get("results", [])))\n    else:\n        entries = []\n    output = []\n    for entry in entries:\n        if not isinstance(entry, dict):\n            continue\n        words = entry.get("words")\n        if isinstance(words, list) and words:\n            output.extend(words)\n        else:\n            output.append(entry)\n    return output\n\ndef align_exact(source_path: str, transcript: str, language: str):\n    if MODEL_ID == "wav2vec2-aligner-zh" and language not in {"zh", "zho", "chi", "cmn", "zh-cn"}:\n        raise HTTPException(status_code=422, detail="the selected Wav2Vec2 aligner supports Mandarin Chinese only")\n    with tempfile.TemporaryDirectory(prefix="la-studio-crisp-align-") as directory:\n        wav_path = str(Path(directory) / "source.wav")\n        output_path = str(Path(directory) / "alignment.json")\n        subprocess.run(\n            ["ffmpeg", "-y", "-v", "error", "-i", source_path, "-vn", "-ac", "1", "-ar", "16000", wav_path],\n            check=True,\n        )\n        command = [\n            CRISPASR, "--align-only", "-am", ALIGNER_MODEL, "-f", wav_path,\n            "--ref-text", transcript, "--align-format", "json",\n            "--align-granularity", "word", "--align-output", output_path,\n            "--gpu-backend", "cuda", "--strict-pipeline", "-l", language,\n        ]\n        result = subprocess.run(command, text=True, capture_output=True)\n        if result.returncode != 0 or not Path(output_path).is_file():\n            raise RuntimeError("CrispASR CUDA aligner failed: " + (result.stderr or result.stdout)[-1600:])\n        payload = json.loads(Path(output_path).read_text(encoding="utf-8"))\n        return _collect_crisp_segments(payload)\n\nTOKEN = os.environ["LA_STUDIO_COLAB_ALIGNMENT_TOKEN"]\nMAX_UPLOAD_BYTES = 512 * 1024 * 1024\nMAX_AUDIO_SECONDS = 300\nALLOWED_CONTENT_TYPES = {\n    "audio/wav", "audio/x-wav", "audio/mpeg", "audio/mp4", "audio/webm",\n    "audio/ogg", "audio/flac", "application/octet-stream",\n}\nALLOWED_EXTENSIONS = {".wav", ".mp3", ".m4a", ".mp4", ".webm", ".ogg", ".flac"}\nREQUEST_SLOTS = threading.BoundedSemaphore(1)\nMODEL_LOCK = threading.Lock()\n\ndef authorize(authorization: str | None) -> None:\n    if authorization != "Bearer " + TOKEN:\n        raise HTTPException(status_code=401, detail="invalid worker token")\n\ndef require_exact_model(requested: str) -> None:\n    if requested.strip().lower() != MODEL_ID:\n        raise HTTPException(\n            status_code=409,\n            detail=f"This worker loaded \'{MODEL_ID}\', but LA Studio requested \'{requested}\'. Open the notebook for the selected model.",\n        )\n\ndef media_duration_seconds(path: str) -> float:\n    probe = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "default=nokey=1:noprint_wrappers=1", path],\n        text=True, capture_output=True,\n    )\n    try:\n        duration = float(probe.stdout.strip())\n    except ValueError:\n        duration = 0.0\n    if probe.returncode != 0 or duration <= 0.0:\n        raise HTTPException(status_code=415, detail="audio is unsupported or could not be decoded")\n    return duration\n\ndef validate_segments(raw_segments) -> list[dict]:\n    segments = []\n    previous_end = 0.0\n    for raw in raw_segments:\n        text = str(raw.get("text", "")).strip()\n        if not text:\n            continue\n        start = float(raw.get("start", raw.get("start_time", 0.0)))\n        end = float(raw.get("end", raw.get("end_time", start)))\n        score = max(0.0, min(1.0, float(raw.get("score", raw.get("confidence", 1.0)))))\n        if start < 0.0 or end < start or start + 0.002 < previous_end:\n            raise RuntimeError("aligner returned non-monotonic timestamps")\n        segments.append({"text": text, "start": start, "end": end, "score": score, "kind": raw.get("kind", "word")})\n        previous_end = end\n    if not segments:\n        raise RuntimeError("aligner returned no timestamped tokens")\n    return segments\n\napp = FastAPI(title=f"LA Studio Alignment - {MODEL_NAME}", docs_url=None, redoc_url=None, openapi_url=None)\n\n@app.get("/health")\n@app.get("/v1/health")\ndef health(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "status": "ready",\n        "ready": True,\n        "device": "cuda",\n        "gpu": torch.cuda.get_device_name(0),\n        "model": MODEL_ID,\n        "upstream_model": UPSTREAM_MODEL,\n        "cpu_fallback": False,\n    }\n\n@app.get("/v1/capabilities")\ndef capabilities(authorization: str | None = Header(default=None)):\n    authorize(authorization)\n    return {\n        "contract_version": 1,\n        "device": "cuda",\n        "capabilities": [{\n            "id": "forced-alignment",\n            "models": [{\n                "id": MODEL_ID,\n                "name": MODEL_NAME,\n                "upstream_model": UPSTREAM_MODEL,\n                "languages": SUPPORTED_LANGUAGES,\n                "max_audio_seconds": MAX_AUDIO_SECONDS,\n                "device": "cuda",\n                "loaded": True,\n            }],\n        }],\n    }\n\n@app.post("/v1/audio/alignments")\nasync def align(\n    audio: UploadFile = File(...),\n    transcript: str = Form(...),\n    language: str = Form("en"),\n    model: str = Form(...),\n    authorization: str | None = Header(default=None),\n):\n    authorize(authorization)\n    require_exact_model(model)\n    text = transcript.strip()\n    if not text:\n        raise HTTPException(status_code=422, detail="transcript is required")\n    suffix = Path(audio.filename or "audio.wav").suffix.lower() or ".wav"\n    if suffix not in ALLOWED_EXTENSIONS:\n        raise HTTPException(status_code=415, detail="unsupported audio filename extension")\n    if audio.content_type and audio.content_type not in ALLOWED_CONTENT_TYPES:\n        raise HTTPException(status_code=415, detail="unsupported audio MIME type")\n    if not REQUEST_SLOTS.acquire(blocking=False):\n        raise HTTPException(status_code=429, detail="the Colab alignment worker is busy; retry shortly")\n    source_path = None\n    try:\n        descriptor, source_path = tempfile.mkstemp(suffix=suffix)\n        with os.fdopen(descriptor, "wb") as output:\n            while chunk := await audio.read(1024 * 1024):\n                output.write(chunk)\n                if output.tell() > MAX_UPLOAD_BYTES:\n                    raise HTTPException(status_code=413, detail="audio exceeds 512 MB upload limit")\n        duration = media_duration_seconds(source_path)\n        if duration > MAX_AUDIO_SECONDS:\n            raise HTTPException(status_code=413, detail="audio exceeds the five minute duration limit")\n        with MODEL_LOCK:\n            raw_segments = align_exact(source_path, text, language.strip().lower() or "en")\n        segments = validate_segments(raw_segments)\n        return {"duration": duration, "segments": segments, "unaligned_tokens": []}\n    except HTTPException:\n        raise\n    except Exception as error:\n        raise HTTPException(\n            status_code=503,\n            detail=f"{MODEL_NAME} alignment failed: {type(error).__name__}: {str(error)[:300]}",\n        ) from error\n    finally:\n        if source_path:\n            Path(source_path).unlink(missing_ok=True)\n        await audio.close()\n        REQUEST_SLOTS.release()\n', encoding='utf-8')
print('Worker source:', WORKER)


In [ ]:
MODEL_ID = 'wav2vec2-aligner-zh'

import os, re, secrets, subprocess, sys, time, urllib.request

TOKEN = secrets.token_urlsafe(32)
env = os.environ.copy()
env["LA_STUDIO_COLAB_ALIGNMENT_TOKEN"] = TOKEN
env["PYTHONUNBUFFERED"] = "1"
worker = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "la_studio_alignment_worker:app", "--host", "127.0.0.1", "--port", "3923"],
    cwd="/content", env=env,
)
for _ in range(240):
    try:
        check = urllib.request.Request(
            "http://127.0.0.1:3923/health",
            headers={"Authorization": "Bearer " + TOKEN},
        )
        with urllib.request.urlopen(check, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError("The exact-model alignment worker did not become ready. Inspect the cell output above.")

subprocess.run(
    ["bash", "-lc", "wget -q -O /content/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i /content/cloudflared.deb"],
    check=True,
)
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3923", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for _ in range(120):
    line = tunnel.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[^\s]+trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError("Cloudflare tunnel URL was not found")

print("\nLA_STUDIO_COLAB_ALIGNMENT_URL=" + public_url)
print("LA_STUDIO_COLAB_ALIGNMENT_TOKEN=" + TOKEN)
print("LA_STUDIO_COLAB_ALIGNMENT_MODEL=" + MODEL_ID)
print("DEVICE=cuda; CPU_FALLBACK=false")
